# 07. Model Explainability: Grad-CAM and ViT Attention Rollout

Generates interpretability heatmaps using Grad-CAM for CNN backbones (Custom CNN, MobileNetV2, ResNet50, EfficientNet-B0, VGG16) and Attention Rollout for Vision Transformers.

In [ ]:
import torch
import matplotlib.pyplot as plt
from cv_benchmark.data import get_dataloaders, CIFAR100_FINE_CLASSES
from cv_benchmark.models import create_model, get_model_target_layer
from cv_benchmark.explainability import GradCAM, overlay_cam_on_image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaders = get_dataloaders(model_name="resnet50", batch_size=4, image_size=(224, 224))
model, _ = create_model("resnet50", num_classes=100, pretrained=True, mode="feature_extraction")
model.to(device)

target_layer = get_model_target_layer("resnet50", model)
gradcam = GradCAM(model, target_layer)

images, targets = next(iter(loaders["test"]))
img_tensor = images[0:1].to(device)
target_cls = int(targets[0].item())

cam = gradcam.generate(img_tensor, target_class=target_cls)
raw_img = images[0].permute(1, 2, 0).numpy()
raw_img = (raw_img - raw_img.min()) / (raw_img.max() - raw_img.min() + 1e-8)
overlay = overlay_cam_on_image(raw_img, cam)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(raw_img)
axes[0].set_title(f"Class: {CIFAR100_FINE_CLASSES[target_cls]}")
axes[1].imshow(cam, cmap="jet")
axes[1].set_title("Grad-CAM Heatmap")
axes[2].imshow(overlay)
axes[2].set_title("Overlay")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()
gradcam.remove_hooks()